# Boston Housing: regresión con dos características
## Entrenamiento, validación y visualización de una superficie de regresión

---

Este notebook estudia un modelo con exactamente dos características de entrada. Esta restricción permite representar los datos y las predicciones mediante una superficie tridimensional: dos ejes para las entradas y un eje para la variable objetivo. Con más de dos características, la gráfica final ya no puede mostrar directamente todo el espacio de entrada.

## 1. Preámbulo/encabezado

In [ ]:
# Manejo y visualización de datos
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Construcción del modelo y división de los datos
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

# Métricas de evaluación
from sklearn.metrics import root_mean_squared_error, r2_score

## 2. Base de datos

La ruta apunta al archivo CSV que contiene Boston Housing. Por facilidad, se asume que `boston.csv` está en el mismo directorio del notebook.

Cada fila representa una observación y cada columna representa una variable. La columna objetivo es `MEDV`, mientras que las demás columnas pueden utilizarse como características de entrada.


In [ ]:
dataset_name = './boston.csv'

## 3. Carga de datos

`pandas.read_csv` carga el archivo directamente en un `DataFrame`. Así se conservan los nombres de las columnas, se facilita la selección de variables y se evitan índices numéricos difíciles de interpretar.


In [ ]:
datos = pd.read_csv(dataset_name)

# Vista inicial de la tabla
datos.head()

Los nombres de las variables se obtienen directamente del `DataFrame`. No es necesario abrir el archivo por separado ni procesar manualmente la primera línea.


In [ ]:
nombres_columnas = datos.columns.tolist()
print("Nombres de columnas:", nombres_columnas)

**Verificación básica de los datos.** Antes de entrenar, conviene revisar el tamaño del conjunto, algunas filas, los tipos de datos y la presencia de valores faltantes. Un modelo puede ejecutarse sin errores y aun así producir resultados incorrectos si los datos fueron cargados o interpretados de forma inadecuada.


In [ ]:
print("Dimensiones (filas, columnas):", datos.shape)
print("\nTipos de datos:")
print(datos.dtypes)
print("\nValores faltantes por columna:")
print(datos.isna().sum())

## 4. Modelo de regresión

### 4.1 División de los datos (entrenamiento/validación)

La función `train_test_split` de scikit-learn separa las entradas y la salida de manera conjunta, por lo que conserva la correspondencia entre cada muestra y su valor objetivo.

Se reserva el 30 % para validación. `random_state` hace reproducible la división: al ejecutar nuevamente el notebook se obtienen los mismos subconjuntos.


In [ ]:
# Proporciones y semilla para una división reproducible
p_test = 0.30
RANDOM_STATE = 42

### 4.2 Selección de las dos características a modelar

Se eligen exactamente dos columnas de entrada mediante sus nombres. Puede cambiar `CARACTERISTICAS`, pero la lista debe conservar dos elementos para que la superficie final pueda representarse en tres dimensiones.

La variable objetivo se mantiene en `MEDV`.


In [ ]:
CARACTERISTICAS = ["RM", "LSTAT"]  # Deben ser exactamente dos
OBJETIVO = "MEDV"

if len(CARACTERISTICAS) != 2:
    raise ValueError("La visualización final requiere exactamente dos características.")

X = datos[CARACTERISTICAS]
y = datos[OBJETIVO]

X_train, X_validation, y_train, y_validation = train_test_split(
    X,
    y,
    test_size=p_test,
    random_state=RANDOM_STATE,
)

print("Características de entrada:", CARACTERISTICAS)
print("Variable de salida:", OBJETIVO)
print("Muestras de entrenamiento:", len(X_train))
print("Muestras de validación:", len(X_validation))


### 4.3 Modelo de regresión

El `Pipeline` aplica primero la transformación polinómica y luego ajusta una regresión lineal. Aunque el estimador final es lineal respecto a sus coeficientes, `PolynomialFeatures` permite modelar curvaturas e interacciones entre las dos entradas.

- `orden = 1`: plano de regresión.
- `orden = 2` o superior: superficie curva.
- `include_bias=False`: el intercepto lo calcula `LinearRegression`.


In [ ]:
# Orden del polinomio
orden = 1  # Modifique este valor para comparar modelos

modelo = make_pipeline(
    PolynomialFeatures(degree=orden, include_bias=False),
    LinearRegression(),
)

# El modelo aprende únicamente a partir del subconjunto de entrenamiento
modelo.fit(X_train, y_train)

### 4.4 Parámetros del modelo

Los coeficientes corresponden a las variables generadas por `PolynomialFeatures`. Por ejemplo, para orden 2 aparecen los términos `RM`, `LSTAT`, `RM²`, `RM·LSTAT` y `LSTAT²`. Mostrar el nombre de cada término evita interpretar los coeficientes solo por su posición.


In [ ]:
polinomio = modelo.named_steps["polynomialfeatures"]
regresion = modelo.named_steps["linearregression"]
nombres_terminos = polinomio.get_feature_names_out(CARACTERISTICAS)

print(f"Orden del polinomio: {orden}")
print(f"Intercepto: {regresion.intercept_:.6f}")
print("Coeficientes:")
for termino, coeficiente in zip(nombres_terminos, regresion.coef_):
    print(f"  {termino}: {coeficiente:.6e}")

### 4.5 Evaluación y validación del modelo

Se emplean métricas de scikit-learn:

- **RMSE:** expresa el tamaño típico del error en las mismas unidades de `MEDV`; valores menores indican predicciones más cercanas.
- **$R^2$:** mide la proporción de variabilidad explicada por el modelo. Un valor cercano a 1 indica mejor ajuste; en validación puede ser negativo si el modelo generaliza mal.

Es importante comparar entrenamiento y validación. Un error muy bajo en entrenamiento acompañado de un error alto en validación puede indicar sobreajuste.

**Desempeño en el conjunto de entrenamiento.** Estas métricas indican qué tan bien se ajusta el modelo a las observaciones utilizadas durante el aprendizaje.


In [ ]:
y_hat_train = modelo.predict(X_train)

rmse_train = root_mean_squared_error(y_train, y_hat_train)
r2_train = r2_score(y_train, y_hat_train)

print(f"RMSE (entrenamiento): {rmse_train:.6f}")
print(f"R²   (entrenamiento): {r2_train:.6f}")

**Desempeño en el conjunto de validación.** Estas observaciones no participaron en el ajuste; por ello ofrecen una estimación más realista del comportamiento del modelo ante datos no vistos.


In [ ]:
y_hat_validation = modelo.predict(X_validation)

rmse_validation = root_mean_squared_error(y_validation, y_hat_validation)
r2_validation = r2_score(y_validation, y_hat_validation)

print(f"RMSE (validación): {rmse_validation:.6f}")
print(f"R²   (validación): {r2_validation:.6f}")

## 5. Visualización de resultados

La gráfica ubica las dos características en los ejes horizontales y `MEDV` en el eje vertical. Los puntos distinguen entrenamiento y validación, mientras que la superficie muestra la predicción del modelo.

La malla se construye dentro del intervalo observado en todos los datos, aunque el modelo se ajustó exclusivamente con entrenamiento. Para predecir, la malla se convierte en un `DataFrame` con los mismos nombres de columnas usados durante el ajuste; esto evita advertencias de scikit-learn sobre nombres de características.

In [ ]:
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection="3d")

# Conversión explícita a NumPy únicamente para las coordenadas del gráfico
ax.scatter(
    X_train[CARACTERISTICAS[0]], X_train[CARACTERISTICAS[1]], y_train,
    color="tab:blue", marker="o", alpha=0.65, label="Entrenamiento"
)
ax.scatter(
    X_validation[CARACTERISTICAS[0]], X_validation[CARACTERISTICAS[1]], y_validation,
    color="tab:red", marker="o", alpha=0.65, label="Validación"
)

# Malla que cubre el intervalo completo observado de ambas características
x1_grid = np.linspace(X[CARACTERISTICAS[0]].min(), X[CARACTERISTICAS[0]].max(), 50)
x2_grid = np.linspace(X[CARACTERISTICAS[1]].min(), X[CARACTERISTICAS[1]].max(), 50)
X1_mesh, X2_mesh = np.meshgrid(x1_grid, x2_grid)

# DataFrame con nombres consistentes con los utilizados durante model.fit(...)
X_grid = pd.DataFrame({
    CARACTERISTICAS[0]: X1_mesh.ravel(),
    CARACTERISTICAS[1]: X2_mesh.ravel(),
})
Y_pred_surface = modelo.predict(X_grid).reshape(X1_mesh.shape)

ax.plot_surface(
    X1_mesh, X2_mesh, Y_pred_surface,
    cmap="viridis", alpha=0.45
)

ax.set_xlabel(CARACTERISTICAS[0])
ax.set_ylabel(CARACTERISTICAS[1])
ax.set_zlabel(OBJETIVO)
ax.set_title(f"Regresión con dos características (polinomio de orden {orden})")
ax.legend()
ax.view_init(elev=10, azim=35)

plt.tight_layout()
plt.show()

---